# 04 - Anomaly Autoencoder

**Author:** Sacha Huberty

**Purpose:** Build a sequential autoencoder over the universe's daily
returns (5-day sequences) to flag anomalous market states from
reconstruction error, cross-check its latent-space GMM clusters
against notebook 03's HMM regimes, then wire the anomaly flag in as a
risk override (blend toward defensive GMV when flagged) and backtest
the resulting strategy OOS against stage 3's baseline.

**Last updated:** 2026-07-25

## Setup

In [ ]:
import copy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.options.display.float_format = '{:.4f}'.format

from atlas import allocation, anomaly, backtest, data, metrics, regimes, strategy, universe

cfg = data.load_config()
posture_cfg = data.load_config(data.PROJECT_ROOT / "config" / "regime_posture.yaml")
cfg["anomaly"]

## Data

In [ ]:
as_of_universe = pd.Timestamp(cfg["general"]["is_end_date"])
universe_df = universe.load_universe(as_of_universe)
tickers = universe_df.index.tolist()
class_bucket = universe_df["class_bucket"]

prices = data.download_prices(tickers, start=cfg["general"]["start_date"])
prices = data.align_calendars(prices)
returns = data.daily_returns(prices)
returns.tail()

## Analysis / signal logic

### Sequential autoencoder (in-sample diagnostic fit)

Fit once on the in-sample period only, exactly like notebook 02's
static snapshot: an illustration of the technique, not a decision (the
OOS backtest below refits independently, on trailing data only, at its
own cadence).

In [ ]:
features = anomaly.build_features(returns.loc[:as_of_universe])
print(f"{len(features)} rows x {features.shape[1]} assets")

ae_result = anomaly.fit_autoencoder(features, cfg)
print(f"Training-error 99th-pct threshold: {ae_result.threshold:.5f}")

In [ ]:
errors = anomaly.reconstruction_error(ae_result, features)
error_dates = features.index[ae_result.seq_len - 1:]
error_series = pd.Series(errors, index=error_dates)
flags = anomaly.is_anomalous(errors, ae_result.threshold)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(error_series.index, error_series.values, linewidth=0.8, label="reconstruction error")
ax.axhline(ae_result.threshold, color="red", linestyle="--", label="99th-pct threshold (train)")
ax.scatter(
    error_series.index[flags], error_series.values[flags],
    color="red", s=12, zorder=3, label="flagged anomaly",
)
ax.set_title("Autoencoder reconstruction error and anomaly flags (in-sample)")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Flagged {flags.sum()} of {len(flags)} days ({flags.mean():.1%})")

### Latent space + GMM regimes vs. the HMM (notebook 03 cross-check)

Do the autoencoder's latent-space clusters agree with the HMM's
regime calls? An independent cross-check, not a second vote -- the
agreement score itself is descriptive, not (yet) part of the
decision.

In [ ]:
latent = anomaly.encode(ae_result, features)
gmm_result = anomaly.fit_latent_gmm(latent, cfg)

fig, ax = plt.subplots(figsize=(7, 6))
scatter = ax.scatter(latent[:, 0], latent[:, 1], c=gmm_result.labels, cmap="viridis", s=10, alpha=0.6)
ax.set_xlabel("Latent dim 1")
ax.set_ylabel("Latent dim 2")
ax.set_title("Autoencoder latent space, colored by GMM cluster")
plt.colorbar(scatter, ax=ax, label="GMM cluster")
plt.show()

In [ ]:
market_ticker = cfg["regimes"]["market_ticker"]
hmm_result = regimes.market_regime(returns.loc[:as_of_universe, market_ticker], cfg, posture_cfg)

hmm_states_aligned = hmm_result["states"].reindex(error_dates)
gmm_labels_aligned = pd.Series(gmm_result.labels, index=error_dates)
valid = hmm_states_aligned.notna()

agreement = anomaly.agreement_score(
    hmm_states_aligned[valid].astype(int).to_numpy(),
    gmm_labels_aligned[valid].to_numpy(),
)
print(f"Adjusted Rand Index, HMM states vs. GMM latent clusters: {agreement:.3f}")
print("(1.0 = perfect agreement up to relabeling, ~0.0 = no better than chance)")

### V1 + anomaly override -> backtest OOS

Wraps stage 3's regime-switching strategy with the anomaly risk
override. Autoencoder refits cost ~7-10s each regardless of epoch
count (mostly fixed Keras build/compile overhead measured directly),
so the lever that matters is refit *count*, not epochs: this backtest
buffers by the HMM's own lookback before the OOS start (same
"buffered by lookback" pattern as stage 2's notebook -- safe now that
`regimes.has_enough_history`/`anomaly.has_enough_history` guard the
cold start) and refits every ~6 months instead of every ~quarter.
Both are explicit stage-4 scope trades for a tractable notebook demo,
revisited under stage 9's real walk-forward infrastructure with
persisted per-fold models.

In [ ]:
oos_start = pd.Timestamp(cfg["general"]["oos_start_date"])
hmm_lookback = cfg["regimes"]["hmm"]["lookback_days"]

buffer_start_pos = max(0, returns.index.searchsorted(oos_start) - hmm_lookback)
backtest_returns = returns.iloc[buffer_start_pos:]

backtest_cfg = copy.deepcopy(cfg)
backtest_cfg["anomaly"]["epochs"] = 10
backtest_cfg["anomaly"]["patience"] = 3
backtest_cfg["anomaly"]["refit_frequency_days"] = 126

base_strategy_fn = strategy.regime_switching_strategy(class_bucket, backtest_cfg, posture_cfg)
anomaly_strategy_fn = strategy.with_anomaly_override(base_strategy_fn, backtest_cfg)

anomaly_result_bt = backtest.run(anomaly_strategy_fn, backtest_returns, backtest_cfg)

In [ ]:
# Stage 2/3 baselines, recomputed here (same buffered range) for a
# direct comparison.
lookback = cfg["optimization"]["lookback_days"]


def make_classical_strategy(method):
    cov_method = cfg["optimization"]["covariance"]

    def strategy_fn(as_of, window):
        w = window.tail(lookback)
        cov_t = allocation.covariance_matrix(w, method=cov_method)
        if method == "risk_parity":
            return allocation.risk_parity(cov_t, cfg)
        raise ValueError(method)
    return strategy_fn


def permanent_strategy(as_of, window):
    return allocation.permanent(class_bucket)


baseline_fns = {
    "permanent": permanent_strategy,
    "risk_parity": make_classical_strategy("risk_parity"),
    "regime_switching": strategy.regime_switching_strategy(class_bucket, cfg, posture_cfg),
}
baseline_results = {
    name: backtest.run(fn, backtest_returns, cfg) for name, fn in baseline_fns.items()
}
all_results = {"regime_switching_anomaly": anomaly_result_bt, **baseline_results}

## Results

In [ ]:
def oos_metrics(result):
    r = result.daily_returns.loc[oos_start:]
    return {
        "ann_return": metrics.ann_return(r),
        "ann_vol": metrics.ann_vol(r),
        "sharpe": metrics.sharpe(r),
        "sortino": metrics.sortino(r),
        "calmar": metrics.calmar(r),
        "max_drawdown": metrics.max_drawdown(r),
        "hit_rate": metrics.hit_rate(r),
        "avg_weekly_turnover": result.turnover.loc[oos_start:].mean(),
        "total_cost_drag": result.costs.loc[oos_start:].sum(),
    }


comparison = pd.DataFrame(
    {name: oos_metrics(res) for name, res in all_results.items()}
).T
comparison.sort_values("sharpe", ascending=False)

In [ ]:
plt.figure(figsize=(11, 6))
for name, res in all_results.items():
    oos_curve = (1.0 + res.daily_returns.loc[oos_start:]).cumprod()
    lw = 2 if name == "regime_switching_anomaly" else 1
    oos_curve.plot(label=name, linewidth=lw)
plt.title("OOS equity curves: regime-switching + anomaly override vs. baselines")
plt.ylabel("Growth of $1, rebased to OOS start")
plt.legend()
plt.show()

## Notes / next steps

- **Honest findings from this run:** the HMM (notebook 03) and the autoencoder's GMM latent clusters show essentially no agreement (Adjusted Rand Index 0.028, vs. 1.0 for perfect and ~0 for chance) -- two independent regime reads that do not coincide here. And the anomaly override barely moved the needle: OOS Sharpe 0.7505 vs. plain regime_switching's 0.7523 (see the comparison table above), essentially unchanged. Anomalies were rare in-sample (2.3% of days flagged) and the override only checks the current day's flag, so its practical impact on this particular backtest was minimal. Per PROJECT_STRUCTURE.md's overfitting defense, that is reported honestly here rather than tuned away; the ablation study in notebook 11 is where the formal marginal-Sharpe verdict lands.

- The autoencoder refit cadence (`anomaly.refit_frequency_days`, ~1
  quarter) and this notebook's lighter OOS-backtest epoch budget are
  both explicit stand-ins for stage 9's real walk-forward
  infrastructure (per-fold persisted models in `models/`), not a
  permanent design choice.
- The HMM/GMM agreement score is descriptive only in this stage: a
  cross-check that the two INDEPENDENT regime reads roughly coincide,
  not (yet) a confidence scaler feeding the decision. Combining them
  into one calibrated confidence signal is natural work for stage 8's
  Black-Litterman view-confidence design.
- Next (stage 5): `meanreversion.py` (ADF, OU half-life, rolling
  z-scores), added as V2, the first per-asset (not just per-posture)
  view.